# 🚗 Advanced Driver Assistance System (ADAS)

## Real-time Object Detection & Lane Recognition using YOLOv8

---

### 📋 Overview

This notebook demonstrates a comprehensive **Advanced Driver Assistance System (ADAS)** that combines:

- **YOLOv8 Object Detection**: Detecting cars, pedestrians, traffic signs, and other objects
- **Lane Detection**: Computer vision-based lane recognition using Canny edge detection and Hough transform
- **Safety Alerts**: Proximity warnings for dangerous objects
- **Real-time Analytics**: Object counting and performance monitoring

### 🎯 Features

- Multi-class object detection (cars, persons, traffic signs)
- detection Lane departure with visual overlays
- Proximity alerts for close objects
- Real-time FPS monitoring
- Interactive visualizations

### 📊 Dataset

This notebook uses the [Car Camera Photos](https://www.kaggle.com/datasets/mbsoroush/car-camera-photos) dataset from Kaggle.

---

## 1. Setup & Dependencies

In [ ]:
# Install required packages
!pip install ultralytics opencv-python-headless numpy pandas matplotlib plotly -q

In [ ]:
import os
import cv2
import glob
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from PIL import Image
from ultralytics import YOLO
import warnings
warnings.filterwarnings('ignore')

print("✅ All dependencies imported successfully!")

## 2. Load Dataset & Model

In [ ]:
# Dataset path (Kaggle's mounted input path)
DATA_PATH = "/kaggle/input/car-camera-photos"

# List available files in dataset
print("📁 Dataset Contents:")
for item in os.listdir(DATA_PATH):
    item_path = os.path.join(DATA_PATH, item)
    if os.path.isdir(item_path):
        files = os.listdir(item_path)
        print(f"  📂 {item}/ ({len(files)} items)")
    else:
        print(f"  📄 {item}")

In [ ]:
# Find image files
image_files = glob.glob(os.path.join(DATA_PATH, "images/images/*.jpg"))
print(f"\n🖼️ Found {len(image_files)} images")

# Display sample image info
if image_files:
    sample_img = cv2.imread(image_files[0])
    print(f"📐 Image shape: {sample_img.shape}")

## 3. Load the Trained Model

In [ ]:
# Load YOLOv8 model (using pretrained for demo - replace with your trained model)
model = YOLO("yolov8n.pt")

print("📦 Model Classes:")
print(model.names)

## 4. Object Detection Showcase

In [ ]:
# Select sample images for detection
sample_images = image_files[:9]  # Take first 9 images

# Create figure for grid display
fig, axes = plt.subplots(3, 3, figsize=(15, 15))
axes = axes.flatten()

detection_stats = []

for idx, img_path in enumerate(sample_images):
    # Read and process image
    frame = cv2.imread(img_path)
    frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    
    # Run detection
    results = model(frame, conf=0.4)
    annotated = results[0].plot()
    
    # Collect statistics
    if results[0].boxes is not None:
        for box in results[0].boxes:
            class_name = model.names[int(box.cls[0])]
            detection_stats.append({
                "image": os.path.basename(img_path),
                "class": class_name,
                "confidence": float(box.conf[0])
            })
    
    # Display
    axes[idx].imshow(annotated)
    axes[idx].axis('off')
    axes[idx].set_title(f"Image {idx+1}", fontsize=10)

plt.tight_layout()
plt.suptitle("🚗 Object Detection Results - Car Camera Photos", fontsize=16, y=1.02)
plt.show()

print("\n✅ Object detection showcase complete!")

## 5. Detection Statistics & Analytics

In [ ]:
# Create DataFrame from detection stats
df = pd.DataFrame(detection_stats)

if not df.empty:
    # Class distribution
    class_counts = df['class'].value_counts()
    
    # Plot class distribution
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Bar chart
    colors = plt.cm.Set3(np.linspace(0, 1, len(class_counts)))
    axes[0].bar(class_counts.index, class_counts.values, color=colors)
    axes[0].set_xlabel('Object Class')
    axes[0].set_ylabel('Count')
    axes[0].set_title('📊 Object Class Distribution')
    axes[0].tick_params(axis='x', rotation=45)
    
    # Pie chart
    axes[1].pie(class_counts.values, labels=class_counts.index, autopct='%1.1f%%', 
                colors=colors, startangle=90)
    axes[1].set_title('🥧 Object Class Proportion')
    
    plt.tight_layout()
    plt.show()
    
    # Display summary
    print("\n📈 Detection Summary:")
    print(f"   Total detections: {len(df)}")
    print(f"   Unique classes: {len(class_counts)}")
    print(f"\n   Class breakdown:")
    for cls, count in class_counts.items():
        print(f"     - {cls}: {count}")
else:
    print("⚠️ No detections found in sample images")

## 6. Interactive Detection Viewer

In [ ]:
# Interactive single image detection
def detect_and_display(image_path, conf=0.4):
    """Detect objects in a single image with detailed output"""
    frame = cv2.imread(image_path)
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    
    # Run detection
    results = model(frame_rgb, conf=conf)
    annotated = results[0].plot()
    
    # Get detection details
    detections = []
    if results[0].boxes is not None:
        for box in results[0].boxes:
            detections.append({
                "class": model.names[int(box.cls[0])],
                "confidence": f"{float(box.conf[0])*100:.1f}%",
                "bbox": [float(x) for x in box.xyxy[0].tolist()]
            })
    
    return annotated, detections

# Process first image as example
sample_path = image_files[0]
annotated_img, detections = detect_and_display(sample_path)

# Display
plt.figure(figsize=(12, 8))
plt.imshow(annotated_img)
plt.axis('off')
plt.title("🔍 Detailed Detection View", fontsize=14)
plt.show()

print(f"\n📋 Detections in first image:")
if detections:
    det_df = pd.DataFrame(detections)
    print(det_df.to_string(index=False))
else:
    print("   No objects detected")

## 7. Lane Detection System

In [ ]:
# Lane Detection Functions

def detect_lane_lines(frame):
    """Detect lane lines using Canny edge detection and Hough transform"""
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (5, 5), 0)
    edges = cv2.Canny(blur, 50, 150)
    
    # Define ROI (Region of Interest) - trapezoid shape for road
    height, width = edges.shape
    roi_points = np.array([
        [
            (int(0.1 * width), height),
            (int(0.45 * width), int(0.6 * height)),
            (int(0.55 * width), int(0.6 * height)),
            (int(0.9 * width), height),
        ]
    ], dtype=np.int32)
    
    mask = np.zeros_like(edges)
    cv2.fillPoly(mask, roi_points, 255)
    cropped_edges = cv2.bitwise_and(edges, mask)
    
    # Hough Transform
    lines = cv2.HoughLinesP(
        cropped_edges,
        rho=1,
        theta=np.pi/180,
        threshold=50,
        minLineLength=40,
        maxLineGap=100
    )
    
    return lines, roi_points


def draw_lane_lines(frame, lanes, color=(128, 0, 128), thickness=3):
    """Draw detected lane lines on the frame"""
    line_image = np.zeros_like(frame)
    
    if lanes is not None:
        for line in lanes:
            x1, y1, x2, y2 = line[0]
            cv2.line(line_image, (x1, y1), (x2, y2), color, thickness)
    
    final_frame = cv2.addWeighted(frame, 0.8, line_image, 1, 1)
    return final_frame

print("✅ Lane detection functions defined!")

In [ ]:
# Lane Detection Showcase
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

for idx, img_path in enumerate(sample_images[:6]):
    row = idx // 3
    col = idx % 3
    
    # Read image
    frame = cv2.imread(img_path)
    
    # Object detection
    results = model(frame, conf=0.4)
    annotated = results[0].plot()
    
    # Lane detection
    lanes, roi = detect_lane_lines(frame)
    final = draw_lane_lines(annotated, lanes)
    
    # Convert to RGB for display
    final_rgb = cv2.cvtColor(final, cv2.COLOR_BGR2RGB)
    
    axes[row, col].imshow(final_rgb)
    axes[row, col].axis('off')
    axes[row, col].set_title(f"Frame {idx+1}", fontsize=11)

plt.suptitle("🛣️ Lane Detection + Object Recognition", fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

print("\n✅ Lane detection showcase complete!")

## 8. Safety Alert System

In [ ]:
# Safety Alert System - Proximity Detection

def safety_alert_system(image_path, proximity_threshold=5000):
    """
    Detect objects and generate safety alerts for close objects
    """
    frame = cv2.imread(image_path)
    
    # Run detection
    results = model(frame, conf=0.4)
    annotated = results[0].plot()
    
    alerts = []
    
    # Check for close objects
    if results[0].boxes is not None:
        for box in results[0].boxes:
            class_id = int(box.cls[0])
            class_name = model.names[class_id]
            
            # Calculate box area (proxy for proximity)
            x1, y1, x2, y2 = box.xyxy[0].tolist()
            box_area = (x2 - x1) * (y2 - y1)
            
            # Alert for large objects (close proximity)
            if box_area > proximity_threshold:
                alerts.append({
                    "class": class_name,
                    "area": int(box_area),
                    "level": "HIGH" if box_area > 10000 else "MEDIUM"
                })
                # Draw warning on image
                cv2.putText(annotated, "⚠️ WARNING: CLOSE OBJECT!", (50, 80),
                           cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 3)
    
    return annotated, alerts

# Test safety alerts on sample images
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

all_alerts = []

for idx, img_path in enumerate(sample_images[:6]):
    row = idx // 3
    col = idx % 3
    
    annotated, alerts = safety_alert_system(img_path)
    all_alerts.extend(alerts)
    
    # Convert to RGB
    annotated_rgb = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)
    
    # Add alert indicator
    if alerts:
        status = f"⚠️ {len(alerts)} Alert(s)"
    else:
        status = "✅ Safe"
    
    axes[row, col].imshow(annotated_rgb)
    axes[row, col].axis('off')
    axes[row, col].set_title(f"Frame {idx+1}: {status}", fontsize=11)

plt.suptitle("🚨 Safety Alert System - Proximity Detection", fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

print(f"\n🚨 Total safety alerts generated: {len(all_alerts)}")

## 9. Performance Metrics

In [ ]:
# Performance Benchmarking

def benchmark_inference(image_paths, model):
    """Benchmark model inference speed"""
    times = []
    
    for img_path in image_paths:
        frame = cv2.imread(img_path)
        
        start_time = time.time()
        results = model(frame, conf=0.4)
        end_time = time.time()
        
        times.append(end_time - start_time)
    
    return {
        "avg_time": np.mean(times),
        "min_time": np.min(times),
        "max_time": np.max(times),
        "fps": 1 / np.mean(times)
    }

# Run benchmark on sample images
benchmark_images = image_files[:20]
metrics = benchmark_inference(benchmark_images, model)

print("⚡ Performance Metrics:")
print(f"   Average inference time: {metrics['avg_time']*1000:.2f} ms")
print(f"   Min inference time: {metrics['min_time']*1000:.2f} ms")
print(f"   Max inference time: {metrics['max_time']*1000:.2f} ms")
print(f"   Estimated FPS: {metrics['fps']:.1f}")

In [ ]:
# Visualize performance
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Inference time distribution
times = []
for img_path in benchmark_images:
    frame = cv2.imread(img_path)
    start = time.time()
    model(frame, conf=0.4)
    times.append(time.time() - start)

axes[0].hist(np.array(times)*1000, bins=10, color='steelblue', edgecolor='black')
axes[0].set_xlabel('Inference Time (ms)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('⏱️ Inference Time Distribution')
axes[0].axvline(np.mean(times)*1000, color='red', linestyle='--', label=f"Mean: {np.mean(times)*1000:.1f}ms")
axes[0].legend()

# FPS visualization
fps_values = [1/t for t in times]
axes[1].bar(range(len(fps_values)), fps_values, color='seagreen', alpha=0.7)
axes[1].axhline(np.mean(fps_values), color='red', linestyle='--', label=f"Avg FPS: {np.mean(fps_values):.1f}")
axes[1].set_xlabel('Frame Index')
axes[1].set_ylabel('FPS')
axes[1].set_title('🚀 Frames Per Second (FPS)')
axes[1].legend()

plt.tight_layout()
plt.show()

## 10. Complete ADAS Pipeline Demo

In [ ]:
# Complete ADAS Pipeline

def adas_pipeline(image_path, conf=0.4, lane_enabled=True, alert_enabled=True):
    """
    Complete ADAS pipeline: Object Detection + Lane Detection + Safety Alerts
    """
    frame = cv2.imread(image_path)
    
    # 1. Object Detection
    results = model(frame, conf=conf)
    annotated = results[0].plot()
    
    # Count detections
    detection_count = {}
    if results[0].boxes is not None:
        for box in results[0].boxes:
            class_name = model.names[int(box.cls[0])]
            detection_count[class_name] = detection_count.get(class_name, 0) + 1
    
    # 2. Lane Detection
    if lane_enabled:
        lanes, _ = detect_lane_lines(frame)
        annotated = draw_lane_lines(annotated, lanes)
    
    # 3. Safety Alerts
    if alert_enabled and results[0].boxes is not None:
        for box in results[0].boxes:
            x1, y1, x2, y2 = box.xyxy[0].tolist()
            box_area = (x2 - x1) * (y2 - y1)
            if box_area > 8000:  # Close object threshold
                class_name = model.names[int(box.cls[0])]
                cv2.rectangle(annotated, (int(x1), int(y1)), (int(x2), int(y2)), (0, 0, 255), 3)
                cv2.putText(annotated, "DANGER!", (int(x1), int(y1)-10),
                           cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)
    
    # 4. Add info overlay
    info_text = f"Objects: {sum(detection_count.values())}"
    cv2.putText(annotated, info_text, (10, 30),
               cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)
    
    return annotated, detection_count

# Run complete pipeline on sample images
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

total_detections = {}

for idx, img_path in enumerate(sample_images[:6]):
    row = idx // 3
    col = idx % 3
    
    result, counts = adas_pipeline(img_path)
    result_rgb = cv2.cvtColor(result, cv2.COLOR_BGR2RGB)
    
    # Aggregate counts
    for k, v in counts.items():
        total_detections[k] = total_detections.get(k, 0) + v
    
    axes[row, col].imshow(result_rgb)
    axes[row, col].axis('off')
    axes[row, col].set_title(f"Frame {idx+1}: {counts}", fontsize=10)

plt.suptitle("🚗 Complete ADAS Pipeline - Object Detection + Lane Recognition + Safety Alerts", 
             fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("\n📊 Total Detections Across All Frames:")
for obj, count in total_detections.items():
    print(f"   {obj}: {count}")

## 11. Conclusion & Summary

In [ ]:
# Summary Statistics
print("=" * 60)
print("📋 ADAS PROJECT SUMMARY")
print("=" * 60)

print("\n🎯 Features Demonstrated:")
print("   ✓ YOLOv8 Object Detection")
print("   ✓ Lane Detection (Canny + Hough Transform)")
print("   ✓ Safety Alert System")
print("   ✓ Real-time Analytics")
print("   ✓ Performance Benchmarking")

print("\n📊 Dataset:")
print(f"   - Source: Kaggle (mbsoroush/car-camera-photos)")
print(f"   - Images processed: {len(sample_images)}")
print(f"   - Total object detections: {sum(total_detections.values())}")

print("\n⚡ Performance:")
print(f"   - Average FPS: {metrics['fps']:.1f}")
print(f"   - Average inference time: {metrics['avg_time']*1000:.2f}ms")

print("\n🔗 Links:")
print("   - GitHub: https://github.com/thetanav/road-vision")
print("   - Dataset: https://www.kaggle.com/datasets/mbsoroush/car-camera-photos")

print("\n" + "=" * 60)
print("✅ Thank you for exploring this ADAS project!")
print("=" * 60)

---

## 📝 Notes

- This notebook uses `yolov8n.pt` (nano) for demonstration
- Replace with your trained `model.pt` for custom object detection
- The dataset is loaded from Kaggle's input path
- All detection thresholds can be adjusted as needed

---
**Created with ❤️ for safer roads and smarter driving**

![https://github.com/thetanav/road-vision](https://img.shields.io/badge/github-adas-blue?style=flat&logo=github)